# Hands on JAX

## 1. How to accumulate values in JAX

In [23]:
### Python way
# A plain Python function that accumulates a sum using a for loop.
# This works fine in regular Python, but won't work inside jax.jit
# because jit needs to trace the computation graph at compile time,
# and a Python for loop with a variable range can't be traced.
def python_sum_of_squares(n):
    total = 0
    for i in range(n):
        total += i ** 2
    return total

print("Python sum of squares:")
print(python_sum_of_squares(10)) 
# Output: 285

Python sum of squares:
285


In [24]:
### Wrong JAX way
# PROBLEM: When jax.jit compiles this function, it replaces 'n' with a
# traced placeholder (a "JitTracer"). Python's range() needs a real integer,
# so it crashes — it can't loop over an abstract placeholder value.
import jax
import jax.numpy as jnp

@jax.jit
def jax_unrolled_loop(n):
    total = jnp.array(0)
    for i in range(n):  # <-- range() needs a concrete Python int, not a tracer
        total += i ** 2
    return total

print("Wrong approach:")
try:
    print(jax_unrolled_loop(10))
except Exception as e:
    print(f"Error: {e}")

Wrong approach:
Error: The __index__() method was called on traced array with shape int32[]
The error occurred while tracing the function jax_unrolled_loop at /var/folders/tw/srcy9s6s1mx8cnq7cj853z4w0000gn/T/ipykernel_50646/2848939173.py:8 for jit. This concrete value was not available in Python because it depends on the value of the argument n.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerIntegerConversionError


In [25]:
### Correct JAX way
# SOLUTION: Use jax.lax.fori_loop instead of Python's range().
# fori_loop is a JAX-native loop — it compiles down to a single XLA op,
# so JAX can trace through it without needing a concrete value for 'n'.
import jax
import jax.numpy as jnp

@jax.jit
def jax_compiled_loop(n):
    # body_fun defines what happens in a single iteration.
    # It must take exactly two arguments: the loop index (i) and the carried state (val).
    def body_fun(i, val):
        return val + i ** 2
    
    # fori_loop(start, stop, body_fn, initial_value)
    # Equivalent to: for i in range(0, n): val = body_fun(i, val)
    return jax.lax.fori_loop(0, n, body_fun, jnp.array(0))

print("Correct approach:")
print(jax_compiled_loop(10))
# Output: 285

Correct approach:
285


## 2. Solving sequential state tracking problems

In [26]:
### Pure Python approach
# This shows the pattern we want to replicate in JAX.
# We loop over a sequence, update a running state each step,
# and collect the state at every step into a history list.
# The key insight: each step depends on the PREVIOUS step's result (sequential).

def python_scan_equivalent(xs, decay=0.9):
    state = 0.0      # Initial carry/state — starts at zero
    history = []     # We dynamically grow this list each step (not allowed in JAX)
    
    for x in xs:
        # Exponential Moving Average: blend old state with new input
        state = decay * state + x
        history.append(state)
        
    return state, history

inputs = [1.0, 2.0, 3.0]
final_val, all_vals = python_scan_equivalent(inputs)
print("Input:  ", inputs)
print("History:", all_vals)
print("Final:  ", final_val)

Input:   [1.0, 2.0, 3.0]
History: [1.0, 2.9, 5.609999999999999]
Final:   5.609999999999999


In [27]:
import jax
import jax.numpy as jnp

# jax.lax.scan is the JAX equivalent of the Python loop above.
# Instead of a growing list, it pre-allocates the output array at compile time.
# It feeds each element of 'xs' through step_fn one at a time,
# threading the updated state (carry) from one step to the next.

@jax.jit
def jax_scan_ema(xs, decay=0.9):
    
    # step_fn MUST take exactly (carry, current_input)
    # and MUST return exactly (new_carry, output_for_this_step)
    def step_fn(carry, x):
        new_state = decay * carry + x  # same EMA formula as the Python version
        # Here the output we want to record IS the new state itself
        return new_state, new_state 
    
    initial_state = 0.0  # same starting point as Python version
    
    # scan(step_fn, initial_carry, sequence)
    # Returns: (final_carry, stacked_outputs_from_every_step)
    final_state, history = jax.lax.scan(step_fn, initial_state, xs)
    
    return final_state, history

inputs = jnp.array([1.0, 2.0, 3.0])
final_val, all_vals = jax_scan_ema(inputs)
print("Input:  ", inputs)
print("History:", all_vals)
print("Final:  ", final_val)

Input:   [1. 2. 3.]
History: [1.   2.9  5.61]
Final:   5.61


## 3. Using vmap to avoid sequential loops

In [28]:
# Python baseline: loop over each row of W and compute a dot product with x.
# This is sequential — one row at a time. Fine for small batches, slow for large ones.
def python_batch_dot(x, W):
    results = []
    for w in W:  # one iteration per row
        dot_product = sum(x_i * w_i for x_i, w_i in zip(x, w))
        results.append(dot_product)
    return results

x = [1.0, 2.0, 3.0]
W = [[4.0, 5.0, 6.0], 
     [7.0, 8.0, 9.0]]

print(python_batch_dot(x, W))
# Output: [32.0, 50.0]

[32.0, 50.0]


In [29]:
import jax
import jax.numpy as jnp

# JAX solution: vmap (vectorized map) eliminates the Python loop entirely.
# Instead of running single_dot_product once per row sequentially,
# vmap transforms it into a batched operation that runs on ALL rows in parallel.

def single_dot_product(x, w):
    # Operates on a single pair: one vector x and one row w
    return jnp.dot(x, w)

@jax.jit
def batched_dot_product(x, W):
    # in_axes=(None, 0) means:
    #   - x is NOT batched (same x is reused for every row)
    #   - W IS batched along axis 0 (iterate over rows)
    return jax.vmap(single_dot_product, in_axes=(None, 0))(x, W)

x_jnp = jnp.array([1.0, 2.0, 3.0])
W_jnp = jnp.array([[4.0, 5.0, 6.0],
                   [7.0, 8.0, 9.0]])

results = batched_dot_product(x_jnp, W_jnp)
print(results)  # [32. 50.]

[32. 50.]


## 4. If else loop - python vs jax

In [30]:
# Python baseline: a simple if/else branch evaluated dynamically per element.
# In JAX, you CANNOT use a Python if/else inside jit on a traced value —
# the condition would need to be a real Python bool, not a traced array.
def python_piecewise(arr):
    result = []
    for x in arr:
        if x > 0:
            result.append(x)
        else:
            result.append(0.0)
    return result

print(python_piecewise([-1.0, 2.0, -3.0, 4.0]))
# Output: [0.0, 2.0, 0.0, 4.0]

[0.0, 2.0, 0.0, 4.0]


In [31]:
## Vectorized masking
# JAX solution for element-wise if/else: jnp.where
# Instead of looping, it evaluates the condition for EVERY element simultaneously
# and picks from the true-branch or false-branch array at each position.
# Think of it as: result[i] = arr[i] if arr[i] > 0 else 0.0

@jax.jit
def jax_where_piecewise(arr):
    # jnp.where(condition, value_if_true, value_if_false)
    # All three arguments are evaluated as full arrays — no branching at runtime
    return jnp.where(arr > 0, arr, 0.0)

arr_jnp = jnp.array([-1.0, 2.0, -3.0, 4.0])
print(jax_where_piecewise(arr_jnp))
# Output: [0. 2. 0. 4.]

[0. 2. 0. 4.]


In [32]:
## jax.lax.cond
# Use jax.lax.cond when you need a TRUE branch on a SINGLE scalar boolean.
# Unlike jnp.where (which is element-wise across arrays),
# lax.cond routes the ENTIRE computation down one of two functions.
#
# Important: BOTH branches are always compiled by JAX — only one runs at runtime.
# This means both branches must accept the same input and return the same shape/dtype.

@jax.jit
def jax_cond_example(x):
    def true_fn(operand):
        return operand * 100.0   # runs if x > 0
        
    def false_fn(operand):
        return operand * -1.0    # runs if x <= 0
        
    # cond(condition, true_fn, false_fn, operand_passed_to_chosen_fn)
    return jax.lax.cond(x > 0, true_fn, false_fn, x)

print(jax_cond_example(jnp.array(5.0)))   # Output: 500.0
print(jax_cond_example(jnp.array(-5.0)))  # Output: 5.0

500.0
5.0


## 5. Conditioning - cell vs row, col

In [33]:
import jax
import jax.numpy as jnp

# This example shows TWO different levels of conditioning:
# A) Element-wise (per cell) — use jnp.where
# B) Row-wise (per whole row) — use jax.lax.cond + jax.vmap

matrix = jnp.array([
    [ 1.0, -2.0,  3.0],  # Row 0 (Sum is positive)
    [-4.0, -5.0, -6.0],  # Row 1 (Sum is negative)
    [ 7.0,  8.0,  0.0]   # Row 2 (Sum is positive)
])

# =====================================================================
# Scenario A: Act on a SINGLE CELL (Element-wise)
# Rule: If a cell is negative, set it to 0.0. Otherwise, keep it.
# jnp.where evaluates the condition for every cell at once — no loop needed.
# =====================================================================
def cell_condition(mat):
    # mat < 0 produces a 3x3 boolean mask, then jnp.where selects per cell
    return jnp.where(mat < 0, 0.0, mat)

print("Cell-by-Cell Output:\n", cell_condition(matrix))


# =====================================================================
# Scenario B: Act on a WHOLE ROW (Vectorized Branching)
# Rule: If the SUM of the row is negative, zero out the ENTIRE row.
# We can't use jnp.where here because the decision is about the whole row,
# not individual cells. So we use lax.cond per row, then vmap across rows.
# =====================================================================
def row_logic(single_row):
    # Reduce the entire row to a single True/False decision
    is_row_negative = jnp.sum(single_row) < 0
    
    def zero_out_row(row):
        return jnp.zeros_like(row)  # replace all values with 0
        
    def keep_row(row):
        return row                  # pass row through unchanged
        
    return jax.lax.cond(is_row_negative, zero_out_row, keep_row, single_row)

# vmap applies row_logic independently to each row (axis 0) of the matrix
apply_to_all_rows = jax.vmap(row_logic, in_axes=0)

print("\nRow-by-Row Output:\n", apply_to_all_rows(matrix))

Cell-by-Cell Output:
 [[1. 0. 3.]
 [0. 0. 0.]
 [7. 8. 0.]]

Row-by-Row Output:
 [[ 1. -2.  3.]
 [ 0.  0.  0.]
 [ 7.  8.  0.]]


## 6. Targeted zeroing

In [34]:
import numpy as np

# Python baseline: directly mutate specific cells in the matrix using index lists.
# rows and cols are paired — so (rows[0], cols[0]), (rows[1], cols[1]), etc.
# are the target positions to zero out.
def python_targeted_zero(matrix, rows, cols, condition):
    if condition:
        matrix[rows, cols] = 0.0  # direct mutation — not allowed in JAX
    return matrix

mat = np.ones((3, 3))
rows = [0, 1, 2]
cols = [2, 0, 1]

print(python_targeted_zero(mat, rows, cols, True))

[[1. 1. 0.]
 [0. 1. 1.]
 [1. 0. 1.]]


In [35]:
import jax
import jax.numpy as jnp

# JAX equivalent of targeted zeroing.
# Key difference: JAX arrays are IMMUTABLE — you can never do matrix[i, j] = value.
# Instead, .at[...].set(...) returns a NEW array with the change applied.
# The original array is untouched.

@jax.jit
def jax_conditional_update(matrix, rows, cols, condition_flag):
    
    def update_fn(mat):
        # .at[rows, cols].set(0.0) — immutable update at multiple positions at once
        # Returns a brand new matrix; the original 'mat' is unchanged
        return mat.at[rows, cols].set(0.0)
        
    def keep_fn(mat):
        return mat  # condition not met — return matrix as-is
        
    # lax.cond routes 'matrix' into either update_fn or keep_fn
    # based on the single boolean condition_flag
    return jax.lax.cond(condition_flag, update_fn, keep_fn, matrix)

mat_jnp = jnp.ones((3, 3))
target_rows = jnp.array([0, 1, 2])
target_cols = jnp.array([2, 0, 1])

print("When Condition is TRUE:")
print(jax_conditional_update(mat_jnp, target_rows, target_cols, True))

print("\nWhen Condition is FALSE:")
print(jax_conditional_update(mat_jnp, target_rows, target_cols, False))

When Condition is TRUE:


[[1. 1. 0.]
 [0. 1. 1.]
 [1. 0. 1.]]

When Condition is FALSE:
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]


## 7. Causal attention masking

In [36]:
import jax
import jax.numpy as jnp

# Causal attention means: token i can only attend to tokens 0..i (past + present).
# It CANNOT look at future tokens (i+1, i+2, ...).
# We enforce this by masking out the upper triangle of the attention score matrix
# with -infinity before softmax, so those positions become 0 probability.

@jax.jit
def causal_attention(q, k):
    # Step 1: Raw attention scores — how much each query "matches" each key
    # Shape: (seq_len, seq_len) — entry [i, j] = how much token i attends to token j
    scores = jnp.matmul(q, k.T) 
    
    N = scores.shape[-1]  # sequence length
    
    # Step 2: Build the causal mask
    # jnp.tril keeps the lower triangle (including diagonal) as True,
    # and sets everything above the diagonal to False.
    # True = "this attention connection is allowed"
    # False = "this is a future token — block it"
    valid_mask = jnp.tril(jnp.ones((N, N), dtype=bool))
    
    # Step 3: Apply the mask
    # Where mask is True  -> keep the real score
    # Where mask is False -> replace with -1e9 (effectively -infinity)
    masked_scores = jnp.where(valid_mask, scores, -1e9)
    
    # Step 4: Softmax turns scores into probabilities.
    # The -1e9 positions become ~0.0, so future tokens get no attention weight.
    attention_weights = jax.nn.softmax(masked_scores, axis=-1)
    
    return attention_weights

seq_len = 4
hidden_dim = 2

q = jnp.ones((seq_len, hidden_dim))
k = jnp.ones((seq_len, hidden_dim))

weights = causal_attention(q, k)

print("Final Attention Weights (Probabilities):")
# Row i shows how token i distributes attention across all tokens.
# Notice token i never attends to tokens after it!
print(jnp.round(weights, 3))

Final Attention Weights (Probabilities):
[[1.    0.    0.    0.   ]
 [0.5   0.5   0.    0.   ]
 [0.333 0.333 0.333 0.   ]
 [0.25  0.25  0.25  0.25 ]]


## 8. Right padding

In [37]:
# Python baseline: pad each sequence to the same length by appending zeros.
# Sequences in a batch can have different lengths — but JAX needs rectangular arrays.
# This padding is done OUTSIDE JAX (on the CPU, before the data enters jit).
def python_right_pad(jagged_batch, max_length, pad_value=0):
    padded_batch = []
    for seq in jagged_batch:
        pad_amount = max_length - len(seq)  # how many zeros to append
        padded_seq = seq + [pad_value] * pad_amount
        padded_batch.append(padded_seq)
    return padded_batch

batch = [
    [1, 2, 3],
    [4, 5, 6, 7, 8],
    [9, 10]
]

print(python_right_pad(batch, max_length=8))

[[1, 2, 3, 0, 0, 0, 0, 0], [4, 5, 6, 7, 8, 0, 0, 0], [9, 10, 0, 0, 0, 0, 0, 0]]


In [38]:
# In the JAX ecosystem, you almost never pad inside the compiled model. 
# You pad in your data loader (using standard NumPy or PyTorch DataLoaders) 
# running on the CPU. By the time the data crosses the boundary 
# into the JAX @jax.jit function, it must already be a perfect, 
# rigid rectangular matrix.

In [39]:
# Because the JAX model receives a perfect rectangle filled with $0$s, 
# we have a new mathematical problem. The Transformer doesn't know 
# that $0$ is a fake word. If we run Self-Attention, the real 
# words will start paying attention to the padding zeros, 
# corrupting the context!Therefore, the JAX code doesn't 
# do the padding; instead, it identifies the padding to 
# mathematically erase it. We generate a Padding Mask 
# using spatial broadcasting.

In [40]:
import jax
import jax.numpy as jnp

# After padding, all sequences are the same rectangular shape —
# but the model doesn't know which tokens are real vs fake (pad) tokens.
# This function creates a boolean mask to identify real tokens,
# which we'll use later to prevent the model from attending to padding.

@jax.jit
def generate_padding_mask(padded_batch, pad_token_id=0):
    # Step 1: For every position, check if it's a real token (not pad_token_id).
    # Result shape: (Batch, Seq_Len) — True = real word, False = padding
    is_real_word = (padded_batch != pad_token_id)
    
    # Step 2: Reshape for use in attention.
    # Attention scores have shape (Batch, Heads, Seq_Len, Seq_Len).
    # We need the mask to broadcast across the query dimension (axis 2),
    # so we insert two extra size-1 dimensions: (Batch, 1, 1, Seq_Len).
    # Broadcasting then applies the mask to every query position automatically.
    attention_mask = is_real_word[:, jnp.newaxis, jnp.newaxis, :]
    
    return attention_mask

padded_input = jnp.array([
    [1, 2, 3, 0, 0],  # 3 real tokens, 2 padding
    [4, 5, 6, 7, 8]   # all real tokens
])

mask = generate_padding_mask(padded_input)

print("Padded Input Shape:", padded_input.shape)
print("Attention Mask Shape:", mask.shape)
# The mask is now ready to be combined with the causal mask!

Padded Input Shape: (2, 5)
Attention Mask Shape: (2, 1, 1, 5)


## 9. Padding mask + combined attention

##### Here we are distributing the attention probabilties to non padded tokens after causal masking

In [41]:
import jax
import jax.numpy as jnp

# Full self-attention combining BOTH masks:
#   1. Causal mask  — token i cannot look at future tokens (geometric constraint)
#   2. Padding mask — no token should attend to pad tokens (data constraint)
# A connection is valid ONLY when BOTH conditions are satisfied.

@jax.jit
def masked_self_attention(q, k, input_ids, pad_id=0):
    # q, k shape: (Batch, Heads, Seq_Len, Head_Dim)
    # input_ids shape: (Batch, Seq_Len)

    # Step 1: Scaled dot-product attention scores
    # Dividing by sqrt(head_dim) prevents scores from getting too large,
    # which would push softmax into a region with near-zero gradients.
    head_dim = q.shape[-1]
    scores = jnp.einsum('bhqd,bhkd->bhqk', q, k) / jnp.sqrt(head_dim)
    
    B, H, L, _ = scores.shape

    # Step 2: Causal mask — lower triangle is True (allowed), upper is False (future)
    causal_mask = jnp.tril(jnp.ones((L, L), dtype=bool))
    
    # Step 3: Padding mask — True where the token is real, False where it's padding
    # Shape (Batch, 1, 1, Seq_Len) so it broadcasts across all query positions
    is_not_pad = (input_ids != pad_id)
    padding_mask = is_not_pad[:, jnp.newaxis, jnp.newaxis, :]
    
    # Step 4: Combine — a connection is valid only if causal AND not padding
    # Broadcasting merges shapes: (L, L) & (Batch, 1, 1, L) -> (Batch, 1, L, L)
    unified_mask = causal_mask & padding_mask
    
    # Step 5: Mask out invalid positions with -1e9 before softmax
    # After softmax, -1e9 becomes ~0.0 — those positions are invisible to the model
    masked_scores = jnp.where(unified_mask, scores, -1e9)
    
    # Step 6: Softmax normalises scores into probabilities along the key dimension
    attention_weights = jax.nn.softmax(masked_scores, axis=-1)
    
    return attention_weights

# Batch of 2 sequences, context length 4
# Sequence 0: 3 real tokens + 1 padding
# Sequence 1: 1 real token  + 3 padding
input_batch = jnp.array([
    [101, 205, 308,   0],
    [101,   0,   0,   0]
])

dummy_q = jnp.ones((2, 1, 4, 2))
dummy_k = jnp.ones((2, 1, 4, 2))

weights = masked_self_attention(dummy_q, dummy_k, input_batch)

print("Attention Probabilities for Sequence 0 (Length 3):")
print(jnp.round(weights[0, 0], 3))

print("\nAttention Probabilities for Sequence 1 (Length 1):")
print(jnp.round(weights[1, 0], 3))

Attention Probabilities for Sequence 0 (Length 3):
[[1.    0.    0.    0.   ]
 [0.5   0.5   0.    0.   ]
 [0.333 0.333 0.333 0.   ]
 [0.333 0.333 0.333 0.   ]]

Attention Probabilities for Sequence 1 (Length 1):
[[1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]]


## 10. Process unique stream

## The `step_fn` logic
For each event:
1. **`is_new`** — look up `seen[uid]`, flip it. True if this user hasn't been counted yet.
2. **`above_thresh`** — `reward > 5.0`.
3. **`should_add`** — both must be true.
4. `jax.lax.cond` routes to `add_fn` (marks user seen, adds reward) or `skip_fn` (passes state unchanged). **Both branches are always compiled** — only one executes at runtime.

---

## Trace through the stream

| Step | uid | reward | is_new | above_thresh | action | total |
|------|-----|--------|--------|--------------|--------|-------|
| 1 | 3 | 8.0 | ✓ | ✓ | add | 8.0 |
| 2 | 1 | 3.0 | ✓ | ✗ | skip | 8.0 |
| 3 | 3 | 10.0 | ✗ | ✓ | skip | 8.0 |
| 4 | 2 | 7.0 | ✓ | ✓ | add | 15.0 |
| 5 | 1 | 6.0 | ✓ | ✓ | add | 21.0 |
| 6 | 4 | 2.0 | ✓ | ✗ | skip | 21.0 |
| 7 | 6 | 9.0 | ✓ | ✓ | add | 30.0 |

In [42]:
import functools
import jax
import jax.numpy as jnp

# Problem: process a live stream of (user_id, reward) events.
# Rules:
#   - Only count a user's reward the FIRST time they appear.
#   - Only count if reward > 5 (threshold filter).
#
# Why static_argnums=(2,)?
#   max_user_id sets the SIZE of the 'seen' boolean array at compile time.
#   JAX needs this to be a concrete Python int (not a traced value),
#   so we mark it static — it won't be traced, just baked into the compiled code.

@functools.partial(jax.jit, static_argnums=(2,))
def aggregate_unique_rewards_dynamic(user_ids, rewards, max_user_id):
    # Pre-allocate a fixed-size boolean array: one slot per possible user ID.
    # Index i = True means user i has already been counted.
    initial_seen  = jnp.zeros(max_user_id + 1, dtype=bool)
    initial_total = jnp.array(0.0)
    initial_carry = (initial_seen, initial_total)

    def step_fn(carry, event):
        seen, total = carry

        # jnp.stack combined user_ids and rewards into float32,
        # so we must cast uid back to int before using it as an index.
        uid    = event[0].astype(jnp.int32)
        reward = event[1]

        # Condition 1: has this user already been counted?
        is_new       = ~seen[uid]          # ~ is bitwise NOT (flips True/False)
        # Condition 2: is the reward above the threshold?
        above_thresh = reward > 5.0
        # Only process if BOTH conditions hold
        should_add   = is_new & above_thresh

        def add_fn(state):
            s, t = state
            # Immutable update: returns a NEW array with seen[uid] = True
            return s.at[uid].set(True), t + reward

        def skip_fn(state):
            # Nothing to do — pass state through unchanged
            return state

        # lax.cond routes carry into add_fn or skip_fn based on should_add.
        # Returns (new_carry, None) — None because we don't need per-step history.
        return jax.lax.cond(should_add, add_fn, skip_fn, carry), None

    # jnp.stack combines the two 1D arrays into a (N, 2) matrix
    # so scan can feed one [uid, reward] row per step.
    final_carry, _ = jax.lax.scan(step_fn, initial_carry, jnp.stack([user_ids, rewards], axis=1))
    final_seen, final_total = final_carry
    return final_seen, final_total

user_ids = jnp.array([3, 1, 3, 2, 1, 4, 6], dtype=jnp.int32)
rewards  = jnp.array([8., 3., 10., 7., 6., 2., 9.])

# int() converts the JAX scalar to a Python int so it can be used as static arg
max_user_id = int(user_ids.max())

seen_users, total_reward = aggregate_unique_rewards_dynamic(user_ids, rewards, max_user_id)
print("Total Reward:", total_reward)   # 30.0  (8 + 7 + 6 + 9)
print("Seen Users:  ", seen_users)     # True at indices 1, 2, 3, 6

Total Reward: 30.0
Seen Users:   [False  True  True  True False False  True]
